# Autor:
ROJAS MARTINEZ JONATHAN FRANCISCO

### Identificación de Neuronas de Entrada y Salida

* **Capa de entrada:** Estará compuesta por **4 neuronas**, ya que el problema define cuatro variables independientes:
  1. Horas de estudio por semana.
  2. Porcentaje de asistencia.
  3. Promedio acumulado.
  4. Número de tareas entregadas.
* **Capa de salida:** Constará de **1 neurona**. Al ser un problema de clasificación binaria (Aprueba o Reprueba), una sola neurona es suficiente para emitir una probabilidad entre 0 y 1.

### Arquitectura de la Red Neuronal Multicapa

Para este problema tabular que no posee una dimensionalidad masiva, se propone una arquitectura tipo "embudo" (*funnel*) que permita extraer patrones sin generar sobreajuste:

* **Número de capas ocultas:** 2 capas ocultas. Una sola capa podría modelar el problema, pero dos capas permiten que la red aprenda relaciones interactivas más complejas.
* **Número de neuronas por capa:**
  * **Primera capa oculta:** 16 neuronas.
  * **Segunda capa oculta:** 8 neuronas.

### Funciones de Activación

* **Capas Ocultas:** Se utilizará la función **ReLU**. 
  * *Justificación:* Es la función estándar más recomendada porque es eficiente computacionalmente, introduce no-linealidad al modelo y evita el problema del vanishing gradient que ocurre con funciones como la tangente hiperbólica al entrenar redes multicapa.
* **Capa de Salida:** Se utilizará la función **Sigmoid**.
  * *Justificación:* En un problema de clasificación binaria, la función sigmoide es obligatoria en la capa de salida porque "aplasta" cualquier valor numérico resultante de la red a un rango estricto entre 0 y 1.

### Comparación y Evaluación con Métricas

Al tratarse de un problema de clasificación binaria, la simple "Exactitud" (Accuracy) no siempre nos dice todo, especialmente si las clases están desbalanceadas (por ejemplo, si aprueban muchos más alumnos de los que reprueban). Las métricas ideales a comparar extraídas de un reporte de clasificación son:

* **Accuracy (Exactitud):** Mide el porcentaje total de predicciones correctas (estudiantes que aprobaban y el modelo dijo que aprobarían, más los que reprobaban y el modelo acertó).
* **Precision (Precisión):** De todos los estudiantes que el modelo predijo que iban a "Aprobar", ¿cuántos aprobaron realmente? Es crucial si queremos evitar falsos positivos.
* **Recall (Sensibilidad):** De todos los estudiantes que "*realmente*" aprobaron en la vida real, ¿cuántos logró detectar el modelo? 
* **F1-Score:** Es la media armónica entre Precisión y Recall. Si este valor es cercano a 1.0, significa que la red neuronal está perfectamente balanceada y no tiene un sesgo hacia ninguna de las dos clases (0 o 1).

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

# Generación de datos sintéticos basados en las 4 variables
# Columnas: [Horas_estudio, Asistencia, Promedio, Tareas_entregadas]
X = np.random.rand(1000, 4) * [40, 100, 10, 20] 
# Regla simple para generar etiquetas: si la suma ponderada es alta, aprueba (1), si no (0)
y = (X[:, 1] * 0.4 + X[:, 2] * 10 > 60).astype(int) 

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Construcción de la Red Neuronal Multicapa
modelo_estudiantes = Sequential([
    Dense(16, input_dim=4, activation='relu', name='Capa_Oculta_1'),
    Dense(8, activation='relu', name='Capa_Oculta_2'),
    Dense(1, activation='sigmoid', name='Capa_Salida')
])

modelo_estudiantes.compile(optimizer='adam', 
                           loss='binary_crossentropy', 
                           metrics=['accuracy'])

print("=== Entrenamiento de la Red ===")
# Entrenamiento de la red
historial = modelo_estudiantes.fit(X_train, y_train, epochs=30, batch_size=32, validation_split=0.2, verbose=0)

# Metricas
y_pred_prob = modelo_estudiantes.predict(X_test)
y_pred_clase = (y_pred_prob > 0.5).astype(int)

print("\n=== Métricas de Evaluación ===")
print(classification_report(y_test, y_pred_clase, target_names=['Reprueba (0)', 'Aprueba (1)']))

c:\Vision-e-Inteligencia-Artificial-2026\.venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


=== Entrenamiento de la Red ===
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 

=== Métricas de Evaluación ===
              precision    recall  f1-score   support

Reprueba (0)       0.99      0.77      0.86        86
 Aprueba (1)       0.85      0.99      0.91       114

    accuracy                           0.90       200
   macro avg       0.92      0.88      0.89       200
weighted avg       0.91      0.90      0.89       200



# Conclusión

Con base a las métricas podemos decir que el modelo es muy bueno para predecir las reprobaciones (90%), por lo cual es sistema puede detectar con mucha precisión a los alumos que deben ayudarles, sin embargo puede ser que algunos aun así reprueben por no ser identificados (1 de cada 10), pero aun así es un modelo que puede ayudar a que menos personas reprueben.